In [ ]:
# Downloads required packages and files
required_files = "https://github.com/jhu-intro-hlt/jhu-intro-hlt.github.io/raw/master/assignments/hw4-files/student/required_files.zip"
! wget $required_files && unzip -o required_files.zip
! pip install -r requirements.txt

--2025-11-17 23:01:03--  https://github.com/jhu-intro-hlt/jhu-intro-hlt.github.io/raw/master/assignments/hw4-files/student/required_files.zip
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/jhu-intro-hlt/jhu-intro-hlt.github.io/master/assignments/hw4-files/student/required_files.zip [following]
--2025-11-17 23:01:03--  https://raw.githubusercontent.com/jhu-intro-hlt/jhu-intro-hlt.github.io/master/assignments/hw4-files/student/required_files.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2282 (2.2K) [application/zip]
Saving to: ‘required_files.zip’

required_files.zip  100%[=================

In [ ]:
# Initialize Otter
import otter

grader = otter.Notebook(colab=True)

# Assignment 4

You have now learnt about end-to-end speech recognition models. In this assignment, you will build a CTC-based end-to-end model for ASR. This assignment is based on the tutorial available [here](https://www.assemblyai.com/blog/end-to-end-speech-recognition-pytorch), and you are encouraged to read through the post and other end-to-end ASR resources available on the internet.

The grading of this assignment is comprised of the successful running of the code with the autograder (30 points) plus the following tiers of performance:
1. +2 points for CER < 0.65, WER < 0.95
2. +5 points for CER < 0.65, WER < 0.9
3. +7 points for CER < 0.55, WER < 0.9
4. +10 points (full credit) for CER < 0.5, WER < 0.8

As per previous assignments, the top 15% of leaderboard will receive 10 extra credit points, and the top 30% will receive 5.

# Setup

For this assignment, as in the previous one, we will be using Google Colab, for both code as well as descriptive questions. Your task is to finish all the questions in the Colab notebook and then upload a PDF version of the notebook, and a viewable link on Gradescope.

### Google colaboratory

Before getting started, get familiar with google colaboratory:
https://colab.research.google.com/notebooks/welcome.ipynb

This is a neat python environment that works in the cloud and does not require you to
set up anything on your personal machine
(it also has some built-in IDE features that make writing code easier).
Moreover, it allows you to copy any existing collaboratory file, alter it and share
with other people.

__Note:__
1. You may need to change your Runtime setting to GPU in order to run the following code blocks.
2. On changing the Runtime setting, you would be required to run the previous code-blocks again.

### Submission

Before you start working on this homework do the following steps:

1. Press __File > Save a copy in Drive...__ tab. This will allow you to have your own copy and change it.
2. Follow all the steps in this collaboratory file and write / change / uncomment code as necessary.
3. Do not forget to occasionally press __File > Save__ tab to save your progress.
4. After all the changes are done and progress is saved press __Share__ button (top right corner of the page), press __get shareable link__ and make sure you have the option __Anyone with the link can view__ selected. Copy the link and paste it in the box below.
5. After completing the notebook, press __File > Download .ipynb__ to download a local copy on your computer, and then upload the file to Gradescope.
6. Please export the notebook to PDF and upload the PDF to the writing part.

__Special handling for model checkpoints.__

7. As the homework requires training neural models, such trained model checkpoints should also be submitted together with the notebook, hence avoiding re-training during the grading phase. For such model checkpoints, they would be stored at `./lightning_logs` directory. You have to first locate the directory from the left side panel (`Files`) on Colab.
8. Enter `./lightning_logs` and find the training label that you would like to submit. The versions are labelled with respect to the training calls.
9. Download the `.ckpt` file from `./lightning_logs/<your_version>/checkpoints/<name>.ckpt`.
10. **Upload the checkpoint file to your JHU OneDrive and get a share link. The permission should be at least `can be viewed by anyone with the link`. Put the link into the corresponding cell.**
11. Submit your notebook to the autograder.

__Paste your notebook link in the box below.__ _(0 points)_

https://colab.research.google.com/drive/1NvAZ7CptwU-4rj_IWiPQND-WkoQBk1cO?usp=sharing

## Installing the requirements

In this assignment, we will use torchaudio for speech feature extraction. You have learned about Mel Frequency Cepstral Coefficients (MFCCs) in the class. [Here](https://haythamfayek.com/2016/04/21/speech-processing-for-machine-learning.html) is another useful blog post describing MFCCs. In the Kaldi tutorial, you extracted MFCCs for ASR, but in this assignment, we will do something slightly different.

First let's set up some helper code and import the libraries which we will use in this experiment.

In [ ]:
import os
from typing import Dict, List, Tuple, Any, Optional, Union
from dataclasses import dataclass

import torch
import torchaudio
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
from transformers.modeling_outputs import CausalLMOutput

try:
    from vocabulary import Vocabulary, encode_as_tensor, decode_as_str
except:
    from files.vocabulary import Vocabulary, encode_as_tensor, decode_as_str

In [ ]:
# Checks whether it is in the autograder grading mode
# Checks whether GPU accelerators are available
is_autograder = os.path.exists('is_autograder.py')
if torch.cuda.is_available() and not is_autograder:
    accelerator = 'gpu'
elif os.environ.get('COLAB_TPU_ADDR') is not None and not is_autograder:
    accelerator = 'tpu'
else:
    accelerator = 'cpu'
print(f'The notebook is running for "{"autograder" if is_autograder else "student"}".')
print('Students should make sure you are running under the "student" mode.')
print(f'You are using "{accelerator}".')

The notebook is running for "student".
Students should make sure you are running under the "student" mode.
You are using "gpu".


In [ ]:
# Seed everything to make sure all experiments are reproducible
pl.seed_everything(seed=777)

INFO:lightning_fabric.utilities.seed:Seed set to 777


777

In [ ]:
# Defines constants
HOMEWORK_DATA_URL = "https://github.com/jhu-intro-hlt/jhu-intro-hlt.github.io/raw/master/assignments/hw4-files/student/"

## Downloading the data

In the Kaldi hands-on session, you worked with the Mini-LibriSpeech data, which is a tiny version of the 960h LibriSpeech (most popular English ASR benchmark dataset). In this assignment, you will use a 100h subset of this data to train your models. Run the code cell below to download the training and testing data.

__Note:__ The training data is approx. 6 GB, and may take several minutes depending on your Internet bandwidth. To avoid downloading the data several times in every session, we download it to Google drive. On running the following code block, you would need to authenticate your Google drive to allow Colab to access its contents. The download may not work if your drive does not have sufficient storage.

In [ ]:
# Attention!!!
# Set this to True, if you are using local machine instead of Colab
RUN_LOCALLY = False
# But you have to make sure that your machine has CUDA support
# Otherwise the training would be super slow.
LOCAL_DATA_STORE_PATH = 'librispeech_data'  # If run locally, please configure a path to store the dataset.
# Only try to use the Colab and Google Drive when not under the autograder environment
if not is_autograder:
    if not RUN_LOCALLY:
        try:
            LIBRISPEECH_DATA_PATH = './gdrive/My Drive/librispeech_data'
            from google.colab import drive

            drive.mount('/content/gdrive')
        except:  # Fall back to local storage
            LIBRISPEECH_DATA_PATH = LOCAL_DATA_STORE_PATH
    else:
        LIBRISPEECH_DATA_PATH = LOCAL_DATA_STORE_PATH
else:
    LIBRISPEECH_DATA_PATH = LOCAL_DATA_STORE_PATH
os.makedirs(LIBRISPEECH_DATA_PATH, exist_ok=True)

Mounted at /content/gdrive


In [ ]:
if not is_autograder:
    train_dataset = torchaudio.datasets.LIBRISPEECH(LIBRISPEECH_DATA_PATH, url='train-clean-100', download=True)
    dev_dataset = torchaudio.datasets.LIBRISPEECH(LIBRISPEECH_DATA_PATH, url='dev-clean', download=True)
else:
    train_dataset = None
    dev_dataset = None

100%|██████████| 5.95G/5.95G [01:27<00:00, 73.2MB/s]
100%|██████████| 322M/322M [00:04<00:00, 77.3MB/s]


# Vocabulary

In this part, we would reuse the `Vocabulary` class from the previous homework (hw3). In order to make the indexing consistent across submissions and models, we create a vocabulary instance here. The only difference is that we added a `<SPACE>` special token.

In [ ]:
char_to_map = char_map_str = """'
a
b
c
d
e
f
g
h
i
j
k
l
m
n
o
p
q
r
s
t
u
v
w
x
y
z"""

vocab = Vocabulary()
for c in char_to_map.split('\n'):
    vocab.add_token(token=c.strip())

## Feature extraction



In the lectures, you have learnt about different methods for feature extraction from audio data. Two of these are:

  1. Mel Frequency Cepstral Coefficients
  2. Mel Spectogram

You used the MFCC features for training in the Kaldi hands-on session. Answer the following questions about these feature extraction methods.

In this assignment, you will use 64-dim Mel Spectogram features for training the models. `torchaudio` makes it easy to extract these features: look at `torchaudio.transforms` for a list of audio transformations available in the library.

Additionally, we use [SpecAugment](https://arxiv.org/pdf/1904.08779.pdf) to add randomness (as a form of data augmentation). This is a simple data augmentation scheme proposed recently that has become very popular since it provides significant WER gains and can be used on-the-fly during training.

Think about what are the 2 main transformations from [this list](https://pytorch.org/tutorials/beginner/audio_preprocessing_tutorial.html#transformations) that can be used to implement SpecAugment.

Complete the following code-block to extract MelSpectogram features with SpecAugment from the training data. For the test data, we just extract MelSpectogram features without any SpecAugment (since data augmentation is only applied on training data).

_Type your answer here, replacing this text._

In [ ]:
# Feature extraction for training data with SpecAugment. Note that Librispeech
# has a sample rate of 16 kHz. Extract 64-dim MelSpectogram features, followed by
# the 2 operations needed for SpecAugment.
# Hint: use `freq_mask_param=30` for frequency masking and `time_mask_param=100`
# for time masking. These specify how much randomness is added to the input
# features.

import torchaudio.transforms as T

# Feature extraction for training data with SpecAugment
train_audio_transforms = nn.Sequential(
    T.MelSpectrogram(
        sample_rate=16000,
        n_mels=64,
        n_fft=400,
        hop_length=160
    ),
    T.FrequencyMasking(freq_mask_param=30),
    T.TimeMasking(time_mask_param=100)
)

# Feature extraction for test data
valid_audio_transforms = T.MelSpectrogram(
    sample_rate=16000,
    n_mels=64,
    n_fft=400,
    hop_length=160
)


In [ ]:
@dataclass
class ASRBatch:
    spectrograms: torch.Tensor  # (batch_size, channel, feature, time)
    labels: torch.Tensor  # (batch_size, label_len, num_labels)
    attention_mask: torch.Tensor  # (batch_size, time)
    label_mask: torch.Tensor  # (batch_size, label_len)
    label_strs: List[str]


class SpeechRecognitionDataModule(pl.LightningDataModule):
    """Wraps PyTorch dataset as a lightning data module."""

    def __init__(
            self,
            datasets: Dict[str, Dataset],
            vocab: Vocabulary,
            batch_size: int = 32,
            shuffle: bool = True
    ):
        super(SpeechRecognitionDataModule, self).__init__()

        self.datasets: Dict[str, Dataset] = {k: v for k, v in datasets.items() if v is not None}
        self.vocab = vocab
        self.batch_size = batch_size
        self.shuffle = shuffle

    @staticmethod
    def _pad_sequence(seq: torch.Tensor, max_length: int, padding_value: Union[int, float, bool]) -> torch.Tensor:
        seq_len = seq.shape[-1]
        if seq_len < max_length:
            return torch.cat(
                [seq, torch.tensor([padding_value] * (max_length - seq_len), dtype=seq.dtype, device=seq.device)],
                dim=-1
            )
        else:
            return seq

    def collate_fn(self, data, phase='train'):
        spectrograms = []
        labels = []
        label_strs = []
        input_lengths = []
        label_lengths = []
        for (waveform, _, utterance, _, _, _) in data:
            if phase == 'train':
                spec = train_audio_transforms(waveform).squeeze(0).transpose(0, 1)
            elif phase == 'val' or phase == 'test':
                spec = valid_audio_transforms(waveform).squeeze(0).transpose(0, 1)
            else:
                raise ValueError
            spectrograms.append(spec)
            label_str = utterance.lower()
            label = encode_as_tensor(self.vocab, label_str).squeeze(0)
            labels.append(label)
            label_strs.append(label_str)
            input_lengths.append(spec.shape[0] // 2)
            label_lengths.append(label.shape[-1])

        spectrograms = nn.utils.rnn.pad_sequence(spectrograms, batch_first=True).unsqueeze(1).transpose(2, 3)
        labels = nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=self.vocab.pad_id())

        max_input_len = max(input_lengths)
        max_label_len = max(label_lengths)
        attention_mask = torch.stack(
            [
                self._pad_sequence(torch.ones(l, dtype=torch.bool), max_length=max_input_len, padding_value=False)
                for l in input_lengths
            ],
            dim=0
        )
        label_mask = torch.stack(
            [
                self._pad_sequence(torch.ones(l, dtype=torch.bool), max_length=max_label_len, padding_value=False)
                for l in label_lengths
            ],
            dim=0
        )

        return ASRBatch(
            spectrograms=spectrograms,
            labels=labels,
            attention_mask=attention_mask,
            label_mask=label_mask,
            label_strs=label_strs
        )

    def train_dataloader(self):
        return DataLoader(self.datasets['train'],
                          batch_size=self.batch_size,
                          shuffle=self.shuffle,
                          collate_fn=lambda x: self.collate_fn(x, phase='train'))

    def val_dataloader(self):
        return DataLoader(self.datasets['val'],
                          batch_size=self.batch_size,
                          shuffle=False,
                          collate_fn=lambda x: self.collate_fn(x, phase='val'))

    def test_dataloader(self):
        return DataLoader(self.datasets['test'],
                          batch_size=self.batch_size,
                          shuffle=False,
                          collate_fn=lambda x: self.collate_fn(x, phase='test'))

In [ ]:
test_datamodule = SpeechRecognitionDataModule(
    datasets={
        'train': train_dataset,
        'val': dev_dataset
    },
    vocab=vocab,
    batch_size=4
)

In [ ]:
train_dataloader = test_datamodule.train_dataloader()
for i, bc in enumerate(train_dataloader):
    if i > 0:
        break
    print(bc)

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

ASRBatch(spectrograms=tensor([[[[5.5423e-04, 2.8045e-04, 2.3831e-04,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [5.1480e-04, 2.4227e-04, 2.0447e-04,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [3.6833e-04, 1.1863e-04, 9.5598e-05,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          ...,
          [4.5311e-07, 5.0684e-07, 4.2597e-07,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [1.4346e-06, 3.7179e-07, 6.2941e-07,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [5.1889e-07, 6.0811e-07, 6.0965e-07,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00]]],


        [[[4.3637e-03, 5.5034e-04, 2.9651e-03,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [3.4564e-03, 4.7133e-04, 3.2641e-03,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [6.8138e-04, 2.1759e-04, 3.8665e-03,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          ...,
          [1.1235e-05, 1.3336e-05, 

In [ ]:
"""Please put your auxiliary modules here."""
...

'Please put your auxiliary modules here.'

In [ ]:
class FeatureExtractor(nn.Module):
    """Extracts features for ASR.

    If needed, you have to use your auxiliary modules to extract features.
    Please modify the `__init__` to make it compatible with your model designs.
    """

    def __init__(
            self,
            input_dim: int = 80,  # e.g., number of mel filterbanks
            hidden_dim: int = 256,
            **kwargs
    ):
        super(FeatureExtractor, self).__init__()

        # Your code here
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=hidden_dim, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """This forward method processes the input spectrogram features and returns
        a tensor that represents transformed features.

        Please pay attention to the output tensor shape, you have to use `transpose`
        to make sure dimensions aligned.

        Parameters
        ----------
        x : torch.Tensor
            The input spectrograms, in the shape of (batch_size, channel, feature, time)

        Returns
        -------
        torch.Tensor
            (batch_size, time, mapped_feature)

        """
        # Your code here
        x = x.squeeze(1)  # Remove the channel dimension -> (batch_size, feature, time)
        x = self.conv1(x)  # Apply Conv1d: (batch_size, hidden_dim, time)
        x = self.relu(x)
        x = self.dropout(x)
        x = x.transpose(1, 2)  # Return to (batch_size, time, hidden_dim)
        return x


class SpeechRecgonitionModel(nn.Module):
    def __init__(
            self,
            vocab: Vocabulary,
            input_dim: int = 80,
            hidden_dim: int = 256,
            rnn_layers: int = 2,
            **kwargs
    ):
        super(SpeechRecgonitionModel, self).__init__()
        self.vocab = vocab

        # In this model, you basically need three modules:
        # 1. A `FeatureExtractor` that extracts features
        # 2. An RNN that processes the sequence
        # 3. A LM head that generates tokens (like in previous homework)

        # Your code here
        num_classes = len(vocab)

        # 1. Feature Extractor
        self.feature_extractor = FeatureExtractor(input_dim=input_dim, hidden_dim=hidden_dim)

        # 2. RNN Layer
        self.rnn = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=rnn_layers, batch_first=True, bidirectional=True)

        # 3. LM Head
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(
            self,
            input_values: Optional[torch.Tensor] = None,
            attention_mask: Optional[torch.Tensor] = None,
            labels: Optional[torch.Tensor] = None,
            label_mask: Optional[torch.Tensor] = None
    ) -> CausalLMOutput:
        """

        Parameters
        ----------
        input_values : Optional[torch.Tensor]
        attention_mask : Optional[torch.Tensor]
        labels : Optional[torch.Tensor]
        label_mask: Optional[torch.Tensor]

        Returns
        -------

        """
        # Your code here
        # You should extract features and compute logits for the sequence.
        features = self.feature_extractor(input_values)

        # Adjust the attention mask to match the time dimension of features
        if attention_mask is not None:
            # Resize the attention mask using interpolation
            time_dim = features.size(1)  # Extract the time dimension after feature extraction
            attention_mask = nn.functional.interpolate(
                attention_mask.unsqueeze(1).float(),  # Add a channel dimension for interpolation
                size=time_dim,
                mode='nearest'  # Nearest neighbor interpolation
            ).squeeze(1).bool()  # Remove the channel dimension and cast back to boolean
            features = features * attention_mask.unsqueeze(-1)

        # Process sequence with RNN
        rnn_out, _ = self.rnn(features)

        # Compute logits
        logits = self.fc(rnn_out)

        loss = None
        if labels is not None:
            target_lengths = label_mask.sum(-1)
            flattened_targets = labels.masked_select(label_mask)
            log_probs = nn.functional.log_softmax(logits, dim=-1).transpose(0, 1)

            # Calculate input_lengths based on attention_mask
            # Since we have kernel_size=3, stride=1, padding=1, the time dimension doesn't change
            if attention_mask is not None:
                input_lengths = attention_mask.sum(-1)
            else:
                input_lengths = torch.full((logits.size(0),), logits.size(1), dtype=torch.long, device=logits.device)

            loss = nn.functional.ctc_loss(
                log_probs=log_probs,
                targets=flattened_targets,
                input_lengths=input_lengths,
                target_lengths=target_lengths,
                blank=self.vocab.pad_id()
            )

        # Please make sure that you are returning a `CausalLMOutput`.
        return CausalLMOutput(
            loss=loss,
            logits=logits
        )

In [ ]:
def wrapped_generate(model_to_wrap: nn.Module, **kwargs) -> torch.Tensor:
    """Wraps the generate method. This function is what will be actually
    called by the evaluation routine for the leaderboard.

    In this function, you can wrap your `generate()` method to allow
    different generation configurations to be used at the test time.

    Parameters
    ----------
    model_to_wrap : nn.Module
        Your ASR model.
    kwargs
        Argument dict that passes all arguments to the generate function.

    Returns
    -------
        Generated phoneme sequences in the form of torch.Tensor.
        In the shape of (batch_size, time).
    """
    # Your code here
    input_values = kwargs.get("input_values")
    attention_mask = kwargs.get("attention_mask")

    # Ensure the model is in evaluation mode
    model_to_wrap.eval()

    # Run the model in no_grad mode
    with torch.no_grad():
        # Forward pass through the model to get logits
        logits = model_to_wrap(input_values=input_values, attention_mask=attention_mask).logits  # (batch_size, time, vocab_size)

        # Greedy decoding: take the most likely token at each timestep
        predicted_ids = logits.argmax(dim=-1)  # (batch_size, time)

    # Post-processing to remove repeated characters and blank tokens (CTC decoding)
    batch_size, seq_len = predicted_ids.shape
    blank_id = model_to_wrap.vocab.pad_id()
    results = []

    for batch_idx in range(batch_size):
        prev_token = None
        decoded_sequence = []

        for token_id in predicted_ids[batch_idx]:
            token_id_item = token_id.item()
            # CTC collapse: remove repeated tokens and blank tokens
            if token_id_item != blank_id and token_id_item != prev_token:
                decoded_sequence.append(token_id_item)
            prev_token = token_id_item

        results.append(decoded_sequence)

    # Convert list of sequences to a padded tensor
    if len(results) > 0 and any(len(seq) > 0 for seq in results):
        max_len = max(len(seq) if len(seq) > 0 else 1 for seq in results)
    else:
        max_len = 1  # Ensure at least 1 column

    # Pad with blank_id to maintain consistency
    padded_results = torch.full((batch_size, max_len), blank_id, dtype=torch.long, device=predicted_ids.device)

    for i, seq in enumerate(results):
        if len(seq) > 0:
            padded_results[i, :len(seq)] = torch.tensor(seq, dtype=torch.long, device=predicted_ids.device)

    return padded_results

In [ ]:
from torchmetrics import CharErrorRate, WordErrorRate
from pytorch_lightning.utilities.types import STEP_OUTPUT


class AutomaticSpeechRecognitionTask(pl.LightningModule):
    def __init__(self,
                 model: nn.Module,
                 vocab: Vocabulary,
                 learning_rate: float = 0.001):
        super(AutomaticSpeechRecognitionTask, self).__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.vocab = vocab

        self.cer = CharErrorRate()
        self.wer = WordErrorRate()

    def training_step(self, batch: ASRBatch) -> STEP_OUTPUT:
        """Defines the training step.

        Parameters
        ----------
        batch : ASRBatch
            The batched training instances.

        Returns
        -------
        loss : torch.Tensor
            The loss computed from the ASR model.
        """
        # Please make necessary modifications to accommodate your design
        outputs = self.model(
            input_values=batch.spectrograms,
            attention_mask=batch.attention_mask,
            labels=batch.labels,
            label_mask=batch.label_mask
        )
        loss = outputs.loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch: ASRBatch, batch_idx: int):
        """Defines the validation step - for this module, we have the same
        training and validation behaviors. Usually, we would compute a metric that is
        used to select the best performing model checkpoint.

        Parameters
        ----------
        batch : ASRBatch
            The batched training instances.
        batch_idx: int
            The index of the batch.
        """
        # You are free to modify this function to ensure they are being called correctly.
        outputs = self.model(
            input_values=batch.spectrograms,
            attention_mask=batch.attention_mask,
        )
        logits = outputs.logits

        # Greedy decoding with CTC collapse (remove repeated tokens and blanks)
        predicted_ids = torch.argmax(logits, dim=-1)  # (batch_size, time)

        # Apply CTC decoding: collapse repeated tokens and remove blanks
        batch_size, seq_len = predicted_ids.shape
        blank_id = self.vocab.pad_id()
        decoded_sequences = []

        for batch_idx in range(batch_size):
            prev_token = None
            decoded_seq = []

            for token_id in predicted_ids[batch_idx]:
                token_id_item = token_id.item()
                # CTC collapse: remove repeated tokens and blank tokens
                if token_id_item != blank_id and token_id_item != prev_token:
                    decoded_seq.append(token_id_item)
                prev_token = token_id_item

            decoded_sequences.append(decoded_seq)

        # Convert decoded sequences to padded tensor for decode_as_str
        max_len = max(len(seq) if len(seq) > 0 else 1 for seq in decoded_sequences)
        padded_predictions = torch.full((batch_size, max_len), blank_id, dtype=torch.long, device=predicted_ids.device)

        for i, seq in enumerate(decoded_sequences):
            if len(seq) > 0:
                padded_predictions[i, :len(seq)] = torch.tensor(seq, dtype=torch.long, device=predicted_ids.device)

        # Decode to strings
        decoded_outputs = decode_as_str(self.vocab, padded_predictions)

        # Compute metrics
        curr_cer = self.cer(decoded_outputs, batch.label_strs)
        curr_wer = self.wer(decoded_outputs, batch.label_strs)

        # Log metrics
        self.log('val_cer', curr_cer, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_wer', curr_wer, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        """Configures optimizers for the training."""
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer

In [ ]:
# Configure data module
asr_datamodule = SpeechRecognitionDataModule(
    datasets={
        'train': train_dataset,
        'val': dev_dataset
    },
    vocab=vocab,
    batch_size=64
)

# Please modify the model and trainer configurations to accommodate your design
asr_model = SpeechRecgonitionModel(
    vocab=vocab,
    input_dim=64,         # Matches the feature dimension from MelSpectrogram
    hidden_dim=256,       # Hidden dimension for feature extraction and RNN
    rnn_layers=2          # Number of RNN layers
)
if is_autograder:  # You have to make sure that you checkpoint can be correctly loaded by the autograder
    # Please upload your checkpoint to your OneDrive
    # and put the share link below `CHECKPOINT_TO_DOWNLOAD`
    CHECKPOINT_TO_DOWNLOAD = 'https://livejohnshopkins-my.sharepoint.com/:u:/g/personal/rchemit1_jh_edu/EV_jT9nAjBNFt-jZRZWUKF4BtKZuNBA6q3HzHUTU1-KHgA?e=tBGtpA'
    # Downloads the checkpoint
    from onedrivedownloader import download as onedrive_download
    onedrive_download(CHECKPOINT_TO_DOWNLOAD, filename='checkpoint.ckpt', unzip=False)
    asr_pl_module = AutomaticSpeechRecognitionTask.load_from_checkpoint(
        checkpoint_path='checkpoint.ckpt',
        vocab=vocab,
        model=asr_model
    )
else:
    # In the student mode, a new model would be trained
    # You are allowed to change training hyperparameters
    # But you are not allowed to create a new task
    asr_pl_module = AutomaticSpeechRecognitionTask(
        vocab=vocab,
        model=asr_model,
        learning_rate=5e-4
    )
    asr_trainer = pl.Trainer(
        accelerator=accelerator,
        min_epochs=10,
        max_epochs=20,
        callbacks=[pl.callbacks.EarlyStopping(monitor='val_cer', mode='min')]
    )
    asr_trainer.fit(model=asr_pl_module, datamodule=asr_datamodule, ckpt_path ='/content/epoch=12-step=5798.ckpt')

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `CharErrorRate` from `torchmetrics` was deprecated and will be removed in 2.0. Import `CharErrorRate` from `torchmetrics.text` instead.
  _future_warning(
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `WordErrorRate` from `torchmetrics` was deprecated and will be removed in 2.0. Import `WordErrorRate` from `torchmetrics.text` instead.
  _future_warning(
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the che

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 64. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Training: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 59. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


In [ ]:
asr_trainer.logged_metrics

{'train_loss_step': tensor(1.0832),
 'val_cer': tensor(0.2907),
 'val_wer': tensor(0.7197),
 'train_loss_epoch': tensor(1.0918)}

In [ ]:
if not is_autograder:
    # You can run this cell to test whether you have get your model trained
    # You are expecting to see a phoneme sequence
    train_dataloader = asr_datamodule.train_dataloader()
    test_example = None
    for i, bc in enumerate(train_dataloader):
        if i > 0:
            break
        test_example = bc

    print(decode_as_str(
        asr_pl_module.model.vocab,
        wrapped_generate(
            model_to_wrap=asr_pl_module.model,
            input_values=test_example.spectrograms.to(asr_pl_module.device),
            attention_mask=test_example.attention_mask.to(asr_pl_module.device)
        )
    ))

['his father wre mostetiv but covatly praut of the etus e is bils an boia', 'pariligin ens an scopes paramont opepation and bitsness parcol and fradmetary atinans', 'the won hi mon for a lat lon and seeme to the gal in a moni', 'ides serea shan jens at', 'that eey might be cald ogod by theis nations which could be boc under him', 'o sea did you ha that the time o hide had with the dicy last wak he askd or noted she toto ad abother in coag eead as eteth you hav anto ma filp', "as if the was no strink tan at ti support the lat of the laber haden hans hischan was fe en aganst is bo an wan icy don pras to him and the acined hand an a shaver he ded not lo got it's ended", 'ii fhaca haves wat you cal my shaht tel agraraf perfected expermentily he xplan rapitly i folstly working oona thet tre clact or theirbots thes naring and soib thess wa carsen cen dendieis is suces', "and o givvi atonic natuly you suffer from the terible ser comstances on sersiman's dath he ton a molans", 'the anjecte suc

In [ ]:
grader.check("model-generate-test")

model-generate-test results: All test cases passed!